# YOLO11s-seg: segmentación de "Pescado" (clase única) — Google Colab (GPU L4)

Entrena un `yolo11s-seg` que segmenta **cualquier pez** (todas las especies → clase `0 = Pescado`) con los tres datasets ya preprocesados (`parana`, `roboflow`, `especies`) en formato YOLO-seg estándar: `dataset_pescado_3merged/{images,labels}/{train,val,test}` + `index.csv`.

**Antes de correrlo en Colab**

1. *Runtime → Change runtime type → GPU L4* (idealmente con RAM alta).
2. Subir a Drive, en `MyDrive/fish_cv_pfc/`, **un solo archivo** `dataset_pescado_3merged.tar` (o `.zip`). Subir 14k archivos sueltos a Drive es lentísimo y leerlos durante el entrenamiento aún más. Se crea en tu PC (los JPG no comprimen, por eso `tar` sin compresión):
   ```powershell
   tar -cf dataset_pescado_3merged.tar dataset_pescado_3merged
   ```
   (También funciona una carpeta ya subida, pero es mucho más lento.) Las listas `.txt` y los `.yaml` del paquete tienen rutas de Windows: **se regeneran** en Colab desde `index.csv`.
3. Correr las celdas en orden.

**Cómo está pensado para Colab**

- El dataset se descomprime al disco local de la VM (`/content`), nunca se entrena leyendo de Drive.
- Los pesos (`last.pt`, `best.pt`) y `results.csv` se copian a Drive **al final de cada época**; si Colab corta la sesión, al volver a correr se retoma solo (`resume`).
- Las secciones de evaluación (3–5) quedan igual; los resultados (csv/png) se guardan también en Drive (sección 6).
- El mismo notebook sigue funcionando en local (si no detecta Colab usa carpetas del repo).

In [ ]:
import gc, os, shutil, subprocess, sys, tarfile, time, warnings, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:   # ultralytics no viene preinstalado en Colab
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics"])

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch
from IPython.display import display
from ultralytics import YOLO

warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------------------- rutas -----------------------------
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/fish_cv_pfc")        # <-- ajusta si lo subiste a otra carpeta
    DRIVE_RUNS = DRIVE_DIR / "runs"
    DRIVE_RUNS.mkdir(parents=True, exist_ok=True)
    ROOT = Path("/content")                                        # disco local de la VM (rápido)
else:
    DRIVE_DIR = DRIVE_RUNS = None
    ROOT = Path.cwd()

DATASET_NAME = "dataset_pescado_3merged"
OUT  = (ROOT / DATASET_NAME).resolve()             # dataset local (formato YOLO-seg)
RUNS = (ROOT / "runs").resolve()                   # salidas del entrenamiento (en Colab se sincronizan a Drive)
RUN_NAME   = "yolo11s-seg_pescado"
BASE_MODEL = "yolo11s-seg.pt"
CLASS_NAME = "Pescado"
SOURCES = ("parana", "roboflow", "especies")
SEED = 42
SPLITS = ["train", "val", "test"]
OVERSAMPLE_PARANA = 3                              # veces que entra cada imagen de train del Paraná por época
RUNS.mkdir(exist_ok=True)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no hay (CPU)")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM libre: {free/1e9:.2f} / {total/1e9:.2f} GB")
print(f"CPUs: {os.cpu_count()} | RAM libre: {psutil.virtual_memory().available/1e9:.0f} GB | "
      f"disco libre: {shutil.disk_usage(ROOT).free/1e9:.0f} GB")

In [ ]:
# Estilo de gráficos: superficie clara, grilla tenue, paleta categórica fija (un color por especie, siempre el mismo)
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
BLUE, ORANGE, AQUA = SERIES[0], SERIES[1], SERIES[2]
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.axisbelow": True, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 10, "axes.titlesize": 11, "legend.frameon": False,
})

## 1. Dataset: copia a disco local y listas de entrenamiento

El preprocesamiento ya dejó el dataset en formato YOLO-seg (`images/<split>`, `labels/<split>`, `index.csv`). Acá solo:

- Se trae de Drive a `/content` (`.tar`/`.zip` → se extrae; carpeta → se copia). Si ya está en el disco local se saltea.
- Se **regeneran** las listas (`train_oversampled.txt`, `val_*.txt`, `test_*.txt`) y los `data*.yaml` con las rutas de esta máquina, a partir de `index.csv`. Se verifica que cada imagen tenga su label.

In [ ]:
def stage_dataset():
    """Trae el dataset de Drive al disco local de la VM (solo en Colab y si no está ya)."""
    if (OUT / "index.csv").exists() and (OUT / "images" / "train").is_dir():
        print("Dataset ya presente en", OUT); return
    if not IN_COLAB:
        raise FileNotFoundError(f"No encuentro {OUT} (corré el preprocesamiento primero)")
    t0 = time.time()
    tar, zp, folder = (DRIVE_DIR / (DATASET_NAME + e) for e in (".tar", ".zip", ""))
    if tar.exists():
        print("Extrayendo", tar.name, "..."); subprocess.check_call(["tar", "-xf", str(tar), "-C", str(ROOT)])
    elif zp.exists():
        print("Extrayendo", zp.name, "..."); subprocess.check_call(["unzip", "-q", "-o", str(zp), "-d", str(ROOT)])
    elif folder.is_dir():
        print("Copiando carpeta desde Drive (lento con muchos archivos; mejor un .tar) ...")
        subprocess.check_call(["cp", "-r", str(folder), str(ROOT)])
    else:
        raise FileNotFoundError(f"Subí {DATASET_NAME}.tar a {DRIVE_DIR}")
    print(f"Listo en {time.time()-t0:.0f}s")

stage_dataset()
df = pd.read_csv(OUT / "index.csv")
assert (OUT / "images" / "train").is_dir() and (OUT / "labels" / "train").is_dir(), "estructura inesperada"

# verifica que cada fila de index.csv tenga imagen y label en disco
falta = [r.name for r in df.itertuples()
         if not (OUT / "images" / r.split / r.name).exists()
         or not (OUT / "labels" / r.split / (Path(r.name).stem + ".txt")).exists()]
if falta:
    print(f"AVISO: {len(falta)} filas sin imagen/label, se ignoran. Ej.: {falta[:3]}")
    df = df[~df.name.isin(falta)].reset_index(drop=True)
print(pd.crosstab(df.source, df.split, margins=True)[SPLITS + ["All"]])

In [ ]:
# Listas por fuente (para validar/testear cada una por separado), train con sobremuestreo del Paraná y yamls
def paths_of(frame):
    return [str(OUT / "images" / r.split / r.name) for r in frame.itertuples()]

for s in SOURCES:
    for sp in ("val", "test"):
        (OUT / f"{sp}_{s}.txt").write_text("\n".join(paths_of(df[(df.source == s) & (df.split == sp)])))

train_paths = paths_of(df[df.split == "train"])
extra = paths_of(df[(df.source == "parana") & (df.split == "train")]) * (OVERSAMPLE_PARANA - 1)
(OUT / "train_oversampled.txt").write_text("\n".join(train_paths + extra))
# val/test combinados como lista explícita (más robusto que la carpeta si hubiera filas sin label)
(OUT / "val_all.txt").write_text("\n".join(paths_of(df[df.split == "val"])))
(OUT / "test_all.txt").write_text("\n".join(paths_of(df[df.split == "test"])))

def write_yaml(fname, **kw):
    body = "".join(f"{k}: {v}\n" for k, v in kw.items())
    (OUT / fname).write_text(f"path: {OUT.as_posix()}\n{body}names:\n  0: {CLASS_NAME}\n")

write_yaml("data.yaml", train="train_oversampled.txt", val="val_all.txt", test="test_all.txt")
for s in SOURCES:
    write_yaml(f"data_{s}.yaml", train="train_oversampled.txt", val=f"val_{s}.txt", test=f"test_{s}.txt")

print(f"Imágenes de train por época: {len(train_paths) + len(extra)} "
      f"(Paraná x{OVERSAMPLE_PARANA}: {(df[(df.source=='parana') & (df.split=='train')]).shape[0] * OVERSAMPLE_PARANA})")
display(df.groupby("source").agg(imagenes=("name", "size"), especies=("species", "nunique"),
                                 instancias_por_img=("n_inst", "mean")).round(2))

## 2. Entrenamiento (yolo11s-seg, Colab L4 24 GB)

Hiperparámetros elegidos y por qué:

| parámetro | valor | motivo |
|---|---|---|
| `imgsz` | 640 | las máscaras de aletas/bigotes necesitan resolución |
| `batch` / `nbs` | 32 / 64 | la L4 tiene 24 GB: el modelo `s` a 640 con AMP usa ~10 GB con batch 32. `nbs=64` acumula 2 pasos, así el paso efectivo sigue siendo 64 y `lr0` estándar vale. Si sobra VRAM se puede subir a 48 |
| `amp` | True | fp16 en tensor cores (la L4 los tiene): ~2× más rápido y la mitad de memoria |
| `cache` | `ram` (o `disk`) | **lo que más acelera en Colab**: evita decodificar JPG grandes en cada época con pocas CPUs. Se elige según la RAM libre |
| `workers` | `min(8, CPUs)` | el mosaic lee 4 imágenes por muestra; más workers que CPUs solo agrega overhead |
| `optimizer` | SGD, `lr0=0.01`, `cos_lr` | con >10k iteraciones es lo que elige `auto`; el coseno evita un `lr` alto en las últimas épocas |
| `epochs` / `patience` | 40 / 10 | el early stopping corta antes si `mAP50-95(M)` se estanca |
| `close_mosaic` | 5 | las últimas 5 épocas sin mosaic para ajustar a imágenes "reales" |
| aug. | mosaic 1.0, fliplr 0.5, `degrees=5`, `scale=0.5`, HSV moderado | el pez aparece en cualquier tamaño/luz; sin flip vertical |
| `single_cls` | True | una sola clase "Pescado" |
| `deterministic` | False | el modo determinista desactiva kernels rápidos de cuDNN |
| `plots` | False | ahorra tiempo y disco; las curvas se grafican abajo |

El train del Paraná entra ×3 por época (ver `train_oversampled.txt`). El `val` del entrenamiento es el val combinado.

**Persistencia:** en cada época `last.pt`/`best.pt`/`results.csv` se copian a `Drive/fish_cv_pfc/runs/<run>/`. Si la sesión se corta, volvé a correr desde la celda de setup y el entrenamiento se retoma.

In [ ]:
BATCH = 32
WORKERS = min(8, os.cpu_count() or 2)

# cache en RAM si entra holgado (~0.9 MB por imagen a 640); si no, en disco local (.npy); si no, sin cache
n_cache = len(train_paths) + len(extra)
need_gb = n_cache * 640 * 480 * 3 / 1e9
avail_gb = psutil.virtual_memory().available / 1e9
if avail_gb > 1.6 * need_gb:
    CACHE = "ram"
elif shutil.disk_usage(ROOT).free / 1e9 > 2 * need_gb:
    CACHE = "disk"
else:
    CACHE = False
print(f"Cache: {CACHE} (necesita ~{need_gb:.0f} GB, RAM libre {avail_gb:.0f} GB)")

HYP = dict(
    epochs=40, patience=10, imgsz=640, batch=BATCH, nbs=64, workers=WORKERS, cache=CACHE,
    optimizer="SGD", lr0=0.01, lrf=0.01, momentum=0.937, weight_decay=5e-4,
    warmup_epochs=2, cos_lr=True, close_mosaic=5,
    mosaic=1.0, mixup=0.0, copy_paste=0.0, fliplr=0.5, flipud=0.0,
    degrees=5, translate=0.1, scale=0.5, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    amp=True, single_cls=True, overlap_mask=True, mask_ratio=4,
    seed=SEED, deterministic=False, plots=False, val=True,
)
FORZAR_ENTRENAR = False      # True = ignora lo que haya en runs/ (local y Drive) y entrena desde cero

# --- sincronización con Drive (Colab) ---
def sync_to_drive(trainer=None):
    """Copia pesos y log del run a Drive. Se llama tras guardar cada época."""
    if not IN_COLAB:
        return
    src_dir, dst_dir = RUNS / RUN_NAME, DRIVE_RUNS / RUN_NAME
    (dst_dir / "weights").mkdir(parents=True, exist_ok=True)
    for rel in ("weights/last.pt", "weights/best.pt", "results.csv", "args.yaml"):
        if (src_dir / rel).exists():
            shutil.copy2(src_dir / rel, dst_dir / rel)

W = RUNS / RUN_NAME / "weights"
last, best_pt = W / "last.pt", W / "best.pt"

# si la VM es nueva pero hay un run en Drive, traerlo para poder retomar
if IN_COLAB and not FORZAR_ENTRENAR and not last.exists() and (DRIVE_RUNS / RUN_NAME / "weights" / "last.pt").exists():
    shutil.copytree(DRIVE_RUNS / RUN_NAME, RUNS / RUN_NAME, dirs_exist_ok=True)
    print("Run recuperado desde Drive")

estado = "nuevo"
if not FORZAR_ENTRENAR and last.exists():
    ep_ckpt = torch.load(last, map_location="cpu", weights_only=False).get("epoch", -1)
    estado = "terminado" if ep_ckpt == -1 else f"interrumpido (época {ep_ckpt + 1})"
print("Estado del entrenamiento:", estado)

if estado == "nuevo":
    t0 = time.time()
    model = YOLO(BASE_MODEL)
    model.add_callback("on_model_save", sync_to_drive)
    model.train(data=str(OUT / "data.yaml"), project=str(RUNS), name=RUN_NAME, exist_ok=True, **HYP)
    print(f"Entrenamiento: {(time.time()-t0)/60:.1f} min | pico VRAM reservada: "
          f"{torch.cuda.max_memory_reserved()/1e9:.2f} GB")
elif estado.startswith("interrumpido"):
    t0 = time.time()
    model = YOLO(last)
    model.add_callback("on_model_save", sync_to_drive)
    model.train(resume=True)
    print(f"Reanudado y terminado en {(time.time()-t0)/60:.1f} min")
else:
    print("Ya hay un entrenamiento terminado; no se re-entrena (FORZAR_ENTRENAR=True para repetirlo).")
sync_to_drive()
gc.collect(); torch.cuda.empty_cache()

In [ ]:
log = pd.read_csv(RUNS / RUN_NAME / "results.csv"); log.columns = log.columns.str.strip()
best_ep = int(log.loc[log["metrics/mAP50-95(M)"].idxmax(), "epoch"])
tr = [c for c in log.columns if c.startswith("train/") and c.endswith("_loss")]
va = [c for c in log.columns if c.startswith("val/") and c.endswith("_loss")]
ep = log["epoch"]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
axes[0].plot(ep, log[tr].sum(axis=1), color=BLUE, label="train"); axes[0].plot(ep, log[va].sum(axis=1), color=ORANGE, label="val"); axes[0].set_title("Loss total")
axes[1].plot(ep, log["train/seg_loss"], color=BLUE, label="train"); axes[1].plot(ep, log["val/seg_loss"], color=ORANGE, label="val"); axes[1].set_title("seg_loss")
axes[2].plot(ep, log["metrics/mAP50(M)"], color=BLUE, label="mAP50"); axes[2].plot(ep, log["metrics/mAP50-95(M)"], color=ORANGE, label="mAP50-95"); axes[2].set_title("mAP máscara (val)")
for ax in axes:
    ax.axvline(best_ep, color=MUTED, ls="--", lw=1, label=f"mejor época ({best_ep})")
    ax.set_xlabel("época"); ax.legend()
plt.tight_layout(); fig.savefig(RUNS / f"curves_{RUN_NAME}.png", dpi=110); plt.show()

## 3. Evaluación

1. Métricas oficiales de Ultralytics (mAP de máscara) en val y test: combinado y por fuente.
2. IoU **por imagen** (unión de las máscaras predichas con confianza ≥ `CONF` contra la unión de las máscaras reales), junto con precisión/recall de píxeles, confianza y propiedades de la imagen (tamaño del pez, contraste con el fondo, nitidez, brillo). Se calcula una vez y se cachea en `runs/<run>_per_image.csv` (se recalcula si cambia `best.pt` o `RECALC=True`).

El `val` se usó para elegir `best.pt` y para el early stopping, así que **el número honesto es el de test**.

In [ ]:
CONF = 0.25          # umbral de confianza para decidir qué máscaras cuentan
IMGSZ = HYP["imgsz"]
HALF = torch.cuda.is_available()
RECALC = False
best = YOLO(best_pt)

res_rows = []
for split in ["val", "test"]:
    for tag, yaml in [("combinado", "data.yaml")] + [(s, f"data_{s}.yaml") for s in SOURCES]:
        m = best.val(data=str(OUT / yaml), split=split, imgsz=IMGSZ, batch=32, workers=WORKERS, half=HALF,
                     project=str(RUNS), name=f"{RUN_NAME}_{split}_{tag}", exist_ok=True, plots=False, verbose=False)
        res_rows.append(dict(split=split, subset=tag, mask_mAP50=m.seg.map50, mask_mAP50_95=m.seg.map,
                             mask_P=m.seg.mp, mask_R=m.seg.mr, box_mAP50=m.box.map50, box_mAP50_95=m.box.map))
map_df = pd.DataFrame(res_rows).round(3)
map_df.to_csv(RUNS / f"results_map_{RUN_NAME}.csv", index=False)
print(f"Parámetros: {sum(p.numel() for p in best.model.parameters())/1e6:.1f} M")
display(map_df)

In [ ]:
def gt_mask(label_path, h, w):
    m = np.zeros((h, w), np.uint8)
    for l in Path(label_path).read_text().splitlines():
        v = l.split()
        if len(v) < 7: continue
        xy = (np.array(v[1:], float).reshape(-1, 2) * [w, h]).round().astype(np.int32)
        cv2.fillPoly(m, [xy], 1)
    return m.astype(bool)

def pred_union(res, conf_thr=CONF):
    h, w = res.orig_shape
    conf = res.boxes.conf.cpu().numpy() if len(res.boxes) else np.zeros(0)
    keep = conf >= conf_thr
    pm = np.zeros((h, w), bool)
    if res.masks is not None and keep.any():
        pm = res.masks.data.cpu().numpy()[keep].any(0)
        if pm.shape != (h, w):
            pm = cv2.resize(pm.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST).astype(bool)
    return pm, conf, int(keep.sum())

def image_features(img_bgr, gm):
    """Propiedades de la imagen (sobre una versión reducida): brillo, contraste, nitidez y contraste pez/fondo."""
    s = 384 / max(img_bgr.shape[:2])
    sm = cv2.resize(img_bgr, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else img_bgr
    g = cv2.resize(gm.astype(np.uint8), (sm.shape[1], sm.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool)
    gray = cv2.cvtColor(sm, cv2.COLOR_BGR2GRAY)
    lab = cv2.cvtColor(sm, cv2.COLOR_BGR2LAB).astype(np.float32)
    bg = float(np.linalg.norm(lab[g].mean(0) - lab[~g].mean(0))) if g.any() and (~g).any() else np.nan
    return dict(brillo=float(gray.mean()), contraste_img=float(gray.std()),
                nitidez=float(cv2.Laplacian(gray, cv2.CV_64F).var()), contraste_pez_fondo=bg)

def analyze(model, frame, imgsz=IMGSZ, half=HALF):
    """IoU y métricas por imagen. `frame` necesita las columnas name, split, source, species."""
    paths = paths_of(frame)
    gen = model.predict(paths, imgsz=imgsz, conf=0.05, retina_masks=True, half=half,
                        stream=True, batch=16, verbose=False)
    out = []
    for r, res in zip(frame.itertuples(), gen):
        h, w = res.orig_shape
        pm, conf, n_det = pred_union(res)
        gm = gt_mask(OUT / "labels" / r.split / (Path(r.name).stem + ".txt"), h, w)
        inter, union = int((pm & gm).sum()), int((pm | gm).sum())
        bx, by, bw, bh = cv2.boundingRect(gm.astype(np.uint8))
        out.append(dict(
            name=r.name, source=r.source, species=r.species, split=r.split,
            iou=inter / max(union, 1), precision=inter / max(int(pm.sum()), 1) if pm.any() else np.nan,
            recall=inter / max(int(gm.sum()), 1), max_conf=float(conf.max()) if len(conf) else 0.0,
            n_det=n_det, n_gt=r.n_inst, gt_area_frac=float(gm.mean()), gt_aspect=bw / max(bh, 1),
            gt_toca_borde=bool(gm[0].any() or gm[-1].any() or gm[:, 0].any() or gm[:, -1].any()),
            ancho=w, alto=h, ms_pre=res.speed["preprocess"], ms_inf=res.speed["inference"],
            ms_post=res.speed["postprocess"], **image_features(res.orig_img, gm)))
    return pd.DataFrame(out)

def muestrear(frame, por, n):
    """n filas al azar (semilla fija) de cada grupo de la columna `por`."""
    return pd.concat([g.sample(min(n, len(g)), random_state=SEED) for _, g in frame.groupby(por)])

STATS_CSV = RUNS / f"{RUN_NAME}_per_image.csv"
if STATS_CSV.exists() and STATS_CSV.stat().st_mtime > best_pt.stat().st_mtime and not RECALC:
    stats = pd.read_csv(STATS_CSV)
    print(f"Cargadas {len(stats)} filas de {STATS_CSV.name}")
else:
    t0 = time.time()
    stats = analyze(best, df[df.split.isin(["val", "test"])])
    stats.to_csv(STATS_CSV, index=False)
    print(f"Analizadas {len(stats)} imágenes en {(time.time()-t0)/60:.1f} min")
gc.collect(); torch.cuda.empty_cache()

### Resumen de IoU por especie, según dataset

`n` = imágenes evaluadas, `IoU_med` = IoU medio, `det%` = % de imágenes donde se detectó algún pez con conf ≥ `CONF`, `IoU<0.5%` = % de imágenes fallidas. Las barras usan **test** (con `val` en la tabla).

In [ ]:
def summarize(g):
    return pd.Series(dict(n=len(g), IoU_med=g.iou.mean(), IoU_mediana=g.iou.median(), IoU_std=g.iou.std(),
                          pct_IoU_menor_05=100 * (g.iou < 0.5).mean(), pct_det=100 * (g.n_det > 0).mean()))

tabla = (stats.groupby(["source", "species", "split"]).apply(summarize, include_groups=False)
         .unstack("split").round(3))
tabla.to_csv(RUNS / f"iou_por_especie_{RUN_NAME}.csv")
pd.set_option("display.max_rows", 200, "display.width", 200)
display(stats.groupby(["source", "split"]).apply(summarize, include_groups=False).round(3))
display(tabla)

# Barras horizontales por fuente (IoU medio en test), ordenadas; opacidad baja si n < 5
te = stats[stats.split == "test"]
fuentes = [s for s in SOURCES if (te.source == s).any()]
alturas = [0.28 * te[te.source == s].species.nunique() + 1.2 for s in fuentes]
fig, axes = plt.subplots(len(fuentes), 1, figsize=(9, sum(alturas)), gridspec_kw=dict(height_ratios=alturas))
for ax, s in zip(np.atleast_1d(axes), fuentes):
    g = te[te.source == s].groupby("species").agg(iou=("iou", "mean"), n=("iou", "size")).sort_values("iou")
    bars = ax.barh(g.index, g.iou, color=BLUE, height=0.7)
    for b, n in zip(bars, g.n):
        if n < 5: b.set_alpha(0.4)
    for y, (v, n) in enumerate(zip(g.iou, g.n)):
        ax.text(v + 0.005, y, f"{v:.3f}  (n={n})", va="center", fontsize=8, color=INK2)
    ax.axvline(te[te.source == s].iou.mean(), color=ORANGE, lw=1.5, ls="--", label=f"media {s}: {te[te.source == s].iou.mean():.3f}")
    ax.set_xlim(0, 1.12); ax.set_title(f"{s} - IoU medio en test por especie"); ax.grid(axis="y", visible=False)
    ax.legend(loc="lower right")
plt.tight_layout(); fig.savefig(RUNS / f"iou_por_especie_{RUN_NAME}.png", dpi=110); plt.show()

## 4. Análisis en profundidad: dataset del Paraná (`dataset/`)

Seis especies: boga, dorado, palometa, sábalo, surubí pintado y tararira. Se analiza (a) cómo son los datos, (b) cómo rinde el modelo por especie, (c) qué tipo de error comete, (d) qué propiedades de la imagen explican los errores, (e) si hay sobreajuste y (f) los casos extremos.

### 4.1 Cómo son los datos

In [ ]:
par = df[df.source == "parana"].copy()
ESP = sorted(par.species.unique())
COL = dict(zip(ESP, SERIES))                         # un color fijo por especie en todo el análisis

# propiedades de cada imagen del Paraná (todos los splits) a partir del label y la cabecera de la imagen
from PIL import Image
def geom(r):
    with Image.open(OUT / "images" / r.split / r.name) as im:
        w, h = im.size
    m = gt_mask(OUT / "labels" / r.split / (Path(r.name).stem + ".txt"), h, w)
    bx, by, bw, bh = cv2.boundingRect(m.astype(np.uint8))
    return dict(ancho=w, alto=h, mpx=w * h / 1e6, gt_area_frac=m.mean(), gt_aspect=bw / max(bh, 1),
                gt_toca_borde=bool(m[0].any() or m[-1].any() or m[:, 0].any() or m[:, -1].any()))
par = pd.concat([par.reset_index(drop=True), pd.DataFrame([geom(r) for r in par.itertuples()])], axis=1)

display(pd.crosstab(par.species, par.split, margins=True)[SPLITS + ["All"]])
display(par.groupby("species").agg(mpx=("mpx", "median"), area_pez=("gt_area_frac", "median"),
                                    aspecto=("gt_aspect", "median"), toca_borde=("gt_toca_borde", "mean"),
                                    inst_por_img=("n_inst", "mean")).round(3))

fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
for ax, (col, ttl) in zip(axes, [("gt_area_frac", "fracción de la imagen ocupada por el pez"),
                                 ("gt_aspect", "aspecto de la caja del pez (ancho/alto)"),
                                 ("mpx", "megapíxeles de la imagen"), ("n_inst", "instancias por imagen")]):
    data = [par[par.species == s][col] for s in ESP]
    bp = ax.boxplot(data, patch_artist=True, widths=0.6, showfliers=False, medianprops=dict(color=INK, lw=1.5))
    for p, s in zip(bp["boxes"], ESP): p.set(facecolor=COL[s], alpha=0.8, edgecolor="none")
    ax.set_xticks(range(1, len(ESP) + 1), ESP, rotation=35, ha="right"); ax.set_title(ttl, fontsize=10)
    ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

In [ ]:
# Muestras del dataset con la máscara real (contorno), 4 por especie
N_MUESTRAS = 4
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(len(ESP), N_MUESTRAS, figsize=(3.6 * N_MUESTRAS, 2.7 * len(ESP)))
for i, s in enumerate(ESP):
    sel = par[par.species == s]
    sel = sel.iloc[rng.choice(len(sel), N_MUESTRAS, replace=False)]
    for j, r in enumerate(sel.itertuples()):
        ax = axes[i, j]; ax.axis("off")
        im = cv2.imread(str(OUT / "images" / r.split / r.name))
        m = gt_mask(OUT / "labels" / r.split / (Path(r.name).stem + ".txt"), *im.shape[:2])
        cs, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(im, cs, -1, (122, 175, 27), max(2, im.shape[1] // 250))      # BGR de AQUA
        ax.imshow(im[..., ::-1])
        if j == 0: ax.set_title(s, loc="left", fontsize=11, color=COL[s])
fig.suptitle("Dataset Paraná - muestras (contorno = máscara real)", fontsize=13)
plt.tight_layout(); plt.show()

### 4.2 Rendimiento por especie (val + test)

In [ ]:
sp_stats = stats[stats.source == "parana"].copy()
sp_stats["detectada"] = sp_stats.n_det > 0

def resumen_especie(g):
    return pd.Series(dict(n=len(g), IoU_med=g.iou.mean(), IoU_mediana=g.iou.median(), IoU_std=g.iou.std(),
                          IoU_p10=g.iou.quantile(0.10), pct_IoU_menor_05=100 * (g.iou < 0.5).mean(),
                          pct_sin_deteccion=100 * (~g.detectada).mean(), precision_px=g.precision.mean(),
                          recall_px=g.recall.mean(), conf_media=g.max_conf.mean(),
                          pct_multi_det=100 * (g.n_det > g.n_gt).mean()))
tab_par = sp_stats.groupby("species").apply(resumen_especie, include_groups=False).round(3)
tab_par.loc["TODAS"] = resumen_especie(sp_stats).round(3)
display(tab_par)
tab_par.to_csv(RUNS / f"parana_resumen_{RUN_NAME}.csv")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw=dict(width_ratios=[1.3, 1]))
ax = axes[0]
data = [sp_stats[sp_stats.species == s].iou for s in ESP]
bp = ax.boxplot(data, patch_artist=True, widths=0.55, showfliers=False, medianprops=dict(color=INK, lw=1.5))
for p, s in zip(bp["boxes"], ESP): p.set(facecolor=COL[s], alpha=0.35, edgecolor=COL[s])
for k, s in enumerate(ESP, 1):
    d = sp_stats[sp_stats.species == s].iou
    ax.scatter(k + rng.uniform(-0.2, 0.2, len(d)), d, s=9, color=COL[s], alpha=0.7, edgecolors="none")
ax.set_xticks(range(1, len(ESP) + 1), ESP); ax.set_ylabel("IoU"); ax.set_title("IoU por imagen y especie (val + test)")
ax.grid(axis="x", visible=False)
ax = axes[1]
for s in ESP:
    d = np.sort(sp_stats[sp_stats.species == s].iou.values)
    ax.plot(d, np.arange(1, len(d) + 1) / len(d), color=COL[s], lw=2, label=s)
ax.set_xlabel("IoU"); ax.set_ylabel("fracción de imágenes con IoU <= x"); ax.set_title("Distribución acumulada de IoU")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

### 4.3 ¿Qué tipo de error?

Cada punto es una imagen. Precisión de píxeles = qué parte de lo predicho es pez; recall = qué parte del pez se encontró.
Debajo de la diagonal el modelo **recorta de menos** (precision > recall: máscara incompleta); sobre la diagonal **se pasa** (recall > precision: incluye fondo/otros objetos). Sin detección → (0, 0).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
ax = axes[0]
for s in ESP:
    d = sp_stats[sp_stats.species == s]
    ax.scatter(d.recall, d.precision.fillna(0), s=14, color=COL[s], alpha=0.65, edgecolors="none", label=s)
ax.plot([0, 1], [0, 1], color=MUTED, lw=1, ls="--")
ax.set_xlabel("recall (píxeles del pez encontrados)"); ax.set_ylabel("precisión (píxeles predichos que son pez)")
ax.set_title("Precisión vs recall por imagen"); ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02); ax.legend(fontsize=8, loc="lower left")

# Tipología de errores por especie (porcentaje de imágenes)
def tipo(r):
    if r.n_det == 0: return "sin detección"
    if r.iou >= 0.9: return "buena (IoU>=0.9)"
    if r.iou >= 0.5:
        return "aceptable (0.5-0.9)"
    return "incompleta (rec<prec)" if r.recall < (r.precision if r.precision == r.precision else 0) else "se pasa (prec<rec)"
sp_stats["tipo"] = sp_stats.apply(tipo, axis=1)
ORDEN = ["buena (IoU>=0.9)", "aceptable (0.5-0.9)", "incompleta (rec<prec)", "se pasa (prec<rec)", "sin detección"]
COLT = [AQUA, BLUE, SERIES[3], ORANGE, "#e34948"]
tip = pd.crosstab(sp_stats.species, sp_stats.tipo, normalize="index").reindex(columns=ORDEN, fill_value=0) * 100
ax = axes[1]
izq = np.zeros(len(tip))
for c, col in zip(ORDEN, COLT):
    ax.barh(tip.index, tip[c], left=izq, color=col, label=c, height=0.7, edgecolor=SURFACE, linewidth=1.5); izq += tip[c].values
ax.set_xlim(0, 100); ax.set_xlabel("% de imágenes"); ax.set_title("Resultado por imagen"); ax.grid(axis="y", visible=False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=8)

ax = axes[2]
cnt = pd.crosstab(sp_stats.species, sp_stats.n_det.clip(upper=3))
cnt.columns = [("3+" if c == 3 else str(c)) for c in cnt.columns]
izq = np.zeros(len(cnt))
for c, col in zip(cnt.columns, [MUTED, BLUE, ORANGE, "#e34948"]):
    ax.barh(cnt.index, cnt[c], left=izq, color=col, label=f"{c} máscaras", height=0.7, edgecolor=SURFACE, linewidth=1.5); izq += cnt[c].values
ax.set_xlabel("imágenes"); ax.set_title("Nº de máscaras predichas (conf >= %.2f)" % CONF); ax.grid(axis="y", visible=False); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
display((tip.round(1)).assign(n=sp_stats.groupby("species").size()))

### 4.4 ¿Qué explica los fallos? Confianza y propiedades de la imagen

Correlación de Spearman entre el IoU y cada propiedad; abajo, dispersión con la mediana por tramo (línea) para ver la forma de la relación.

In [ ]:
FEATS = {"max_conf": "confianza máxima", "gt_area_frac": "fracción de imagen que ocupa el pez",
         "contraste_pez_fondo": "contraste pez/fondo (ΔE Lab)", "nitidez": "nitidez (var. Laplaciano)",
         "brillo": "brillo medio", "contraste_img": "contraste de la imagen", "gt_aspect": "aspecto de la caja (ancho/alto)",
         "ancho": "ancho (px)"}
rho = pd.DataFrame({f: [sp_stats[[f, "iou"]].corr(method="spearman").iloc[0, 1]] +
                       [sp_stats[sp_stats.species == s][[f, "iou"]].corr(method="spearman").iloc[0, 1] for s in ESP]
                    for f in FEATS}, index=["TODAS"] + ESP).T.round(2)
display(rho.rename(index=FEATS))

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, f in zip(axes.ravel(), ["max_conf", "gt_area_frac", "contraste_pez_fondo", "nitidez", "brillo", "gt_aspect"]):
    for s in ESP:
        d = sp_stats[sp_stats.species == s]
        ax.scatter(d[f], d.iou, s=10, color=COL[s], alpha=0.5, edgecolors="none", label=s)
    x = sp_stats[f]
    if f == "nitidez": ax.set_xscale("log")
    tramos = pd.qcut(x.rank(method="first"), 8)
    med = sp_stats.groupby(tramos, observed=True).agg(x=(f, "median"), y=("iou", "median"))
    ax.plot(med.x, med.y, color=INK, lw=2.2, marker="o", ms=5, label="mediana por tramo")
    ax.set_xlabel(FEATS[f]); ax.set_ylabel("IoU"); ax.set_ylim(-0.02, 1.02)
    ax.set_title(f"Spearman = {rho.loc[f, 'TODAS']:+.2f}", fontsize=10)
axes[0, 0].legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.show()

### 4.5 ¿Hay sobreajuste? Train vs. val/test del Paraná

Se evalúan 40 imágenes de **train** por especie (muestra fija) y se comparan con val+test. Una brecha grande indicaría que el modelo memorizó esas 913 imágenes (que entran ×3 por época).

In [ ]:
tr_sample = muestrear(par[par.split == "train"], "species", 40)
tr_stats = analyze(best, tr_sample[["name", "split", "source", "species", "n_inst"]])
gap = pd.DataFrame({"IoU_train": tr_stats.groupby("species").iou.mean(),
                    "IoU_val+test": sp_stats.groupby("species").iou.mean()})
gap["brecha"] = gap.IoU_train - gap["IoU_val+test"]
gap.loc["TODAS"] = [tr_stats.iou.mean(), sp_stats.iou.mean(), tr_stats.iou.mean() - sp_stats.iou.mean()]
display(gap.round(3))

fig, ax = plt.subplots(figsize=(8, 3.8))
y = np.arange(len(ESP))
ax.barh(y - 0.2, gap.loc[ESP, "IoU_train"], height=0.38, color=BLUE, label="train (muestra)")
ax.barh(y + 0.2, gap.loc[ESP, "IoU_val+test"], height=0.38, color=ORANGE, label="val + test")
ax.set_yticks(y, ESP); ax.set_xlim(max(0, gap[["IoU_train", "IoU_val+test"]].min().min() - 0.1), 1.0)
ax.invert_yaxis(); ax.set_xlabel("IoU medio"); ax.grid(axis="y", visible=False); ax.legend(loc="lower right")
ax.set_title("Brecha train vs val+test (Paraná)")
plt.tight_layout(); plt.show()

### 4.6 Casos extremos: top 20 IoU, bottom 20 IoU y bottom 20 confianza

Contorno aqua = máscara real; relleno naranja = máscara predicha (conf ≥ `CONF`). Se usan las imágenes del Paraná de **val + test**.

In [ ]:
def show_cases(frame, title, fname, cols=5):
    n = len(frame); rows_ = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows_, cols, figsize=(3.7 * cols, 3.2 * rows_))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes: ax.axis("off")
    for ax, s in zip(axes, frame.itertuples()):
        im = cv2.imread(str(OUT / "images" / s.split / s.name))
        res = best.predict(im, imgsz=IMGSZ, conf=0.05, retina_masks=True, half=HALF, verbose=False)[0]
        pm, _, _ = pred_union(res)
        gm = gt_mask(OUT / "labels" / s.split / (Path(s.name).stem + ".txt"), *im.shape[:2])
        ov = im.copy(); ov[pm] = (52, 104, 235)                                      # BGR de ORANGE
        im = cv2.addWeighted(ov, 0.5, im, 0.5, 0)
        cs, _ = cv2.findContours(gm.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(im, cs, -1, (122, 175, 27), max(2, im.shape[1] // 200))     # BGR de AQUA
        ax.imshow(im[..., ::-1])
        ax.set_title(f"{s.species} ({s.split}) | IoU {s.iou:.2f} | conf {s.max_conf:.2f}", fontsize=9)
    fig.suptitle(title, fontsize=14)
    plt.tight_layout(); fig.savefig(RUNS / fname, dpi=100); plt.show()

N_CASOS = 20
show_cases(sp_stats.sort_values("iou", ascending=False).head(N_CASOS),
           f"Paraná - top {N_CASOS} IoU", f"parana_top_iou_{RUN_NAME}.png")
show_cases(sp_stats.sort_values("iou").head(N_CASOS),
           f"Paraná - bottom {N_CASOS} IoU", f"parana_bottom_iou_{RUN_NAME}.png")
con_det = sp_stats[sp_stats.max_conf > 0]
print(f"Imágenes sin ninguna detección (conf 0.05): {(sp_stats.max_conf == 0).sum()} de {len(sp_stats)} (no entran al ranking de confianza)")
show_cases(con_det.sort_values("max_conf").head(N_CASOS),
           f"Paraná - bottom {N_CASOS} confianza", f"parana_bottom_conf_{RUN_NAME}.png")

In [ ]:
# Lectura rápida automática del análisis
peor = tab_par.drop("TODAS").IoU_med.idxmin(); mejor = tab_par.drop("TODAS").IoU_med.idxmax()
r_all = rho["TODAS"].drop("ancho").abs().sort_values(ascending=False)
print(f"- IoU medio Paraná (val+test): {sp_stats.iou.mean():.3f} | mediana {sp_stats.iou.median():.3f} | "
      f"{100*(sp_stats.iou < 0.5).mean():.1f}% de imágenes con IoU<0.5")
print(f"- Mejor especie: {mejor} ({tab_par.loc[mejor, 'IoU_med']:.3f}) | peor: {peor} ({tab_par.loc[peor, 'IoU_med']:.3f})")
print(f"- Propiedad más ligada al IoU: {FEATS[r_all.index[0]]} (|rho|={r_all.iloc[0]:.2f}); "
      f"luego {FEATS[r_all.index[1]]} (|rho|={r_all.iloc[1]:.2f})")
print(f"- Recall medio {sp_stats.recall.mean():.3f} vs precisión media {sp_stats.precision.mean():.3f} -> "
      f"el modelo tiende a {'recortar de menos' if sp_stats.recall.mean() < sp_stats.precision.mean() else 'pasarse'}")
print(f"- Brecha train - val/test: {gap.loc['TODAS', 'brecha']:+.3f} IoU")

## 5. Velocidad de inferencia en la L4 de Colab

Latencia por imagen (preproceso + inferencia + postproceso, ms) y IoU en una muestra fija de val+test, variando `imgsz` y fp16. Sirve para decidir si conviene inferir a una resolución menor que la de entrenamiento.

In [ ]:
muestra = muestrear(df[df.split.isin(["val", "test"])], "source", 100)
best.predict(paths_of(muestra)[:8], imgsz=IMGSZ, half=HALF, verbose=False)       # calentamiento

bench = []
for sz in (480, 544, 640, 736):
    for half in ([True, False] if HALF else [False]):
        if sz != IMGSZ and not half: continue
        a = analyze(best, muestra, imgsz=sz, half=half)
        bench.append(dict(imgsz=sz, fp16=half, IoU=a.iou.mean(), ms_total=(a.ms_pre + a.ms_inf + a.ms_post).mean(),
                          ms_inferencia=a.ms_inf.mean(), ms_postproceso=a.ms_post.mean()))
bench = pd.DataFrame(bench); bench["FPS"] = 1000 / bench.ms_total
display(bench.round(3))
# Para más velocidad: best.export(format="engine", half=True, imgsz=IMGSZ)   (requiere TensorRT)
#                     best.export(format="onnx", half=True, imgsz=IMGSZ)
print("Pesos finales:", best_pt)

## 6. Guardar resultados en Drive

Copia los csv/png de análisis (`runs/*.csv`, `runs/*.png`) y los pesos finales a `Drive/fish_cv_pfc/runs/`.

In [ ]:
if IN_COLAB:
    sync_to_drive()
    for f in list(RUNS.glob("*.csv")) + list(RUNS.glob("*.png")):
        shutil.copy2(f, DRIVE_RUNS / f.name)
    print("Copiado a", DRIVE_RUNS, "->", sorted(p.name for p in DRIVE_RUNS.iterdir()))
else:
    print("Ejecución local: nada que sincronizar.")